# H2: EMA 200 Attraction After SSL Crossover

After both SSL(60) and SSL(120) regimes align on a direction, does price move toward the EMA 200 cloud?

**Crossover definition:** Both SSL regimes agree on direction , either both flip bullish or both flip bearish simultaneously.

**Reach definition:** After Bullish crossover - any bar's high touches the lower cloud boundary (EMA close). After Bearish crossover - any bar's low touches the upper cloud boundary (EMA high).

**Controls:**
- **Control A (Random):** Same measurement from random bars, direction determined by position relative to EMA.
- **Control B (Momentum-matched):** Non-crossover bars with similar sized price moves (±20% tolerance). Tests whether crossovers are special or just detect big moves.

In [ ]:
import sys
sys.path.append('../src')
from tests import h2_test
from tests import h2_control_b
from aggregation_data import load_and_resample

# BTC/USD data
btc_4h = load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '4h')
timeframes = {
    '15min': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '15min'),
    '30min': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '30min'),
    '1h': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '1h'),
    '2h': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '2h'),
    '4h': btc_4h
}
# EUR/USD data
eur_4h = load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '4h')
eur_timeframes = {
    '15min': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '15min'),
    '30min': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '30min'),
    '1h': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '1h'),
    '2h': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '2h'),
    '4h': eur_4h
}

## BTC FUTURES(CME) - CONTROL A (vs Random Bars)

In [ ]:
for tf_name, df in timeframes.items():
    for mb in [10, 20, 30]:
        real_rate, random_rate, z_score, z_p, mw_p, avg_bars = h2_test(df, max_bars=mb)
        print(f"{tf_name} mb={mb}: real={real_rate:.3f}, random={random_rate:.3f}, z={z_score:.2f}, z_p={z_p:.4f}, mw_p={mw_p:.4f}, avg_bars={avg_bars:.1f}")

### BTC Control A Results

| Timeframe | Max Bars | Crossover Rate | Random Rate | Z-score | Z p-value | MW p-value | Avg Bars |
|-----------|----------|---------------|-------------|---------|-----------|------------|----------|
| 15min | 10 | 0.663 | 0.270 | 51.24 | <0.0001 | <0.0001 | 1.7 |
| 15min | 20 | 0.714 | 0.377 | 41.17 | <0.0001 | <0.0001 | 2.6 |
| 15min | 30 | 0.757 | 0.465 | 35.15 | <0.0001 | <0.0001 | 3.9 |
| 30min | 10 | 0.677 | 0.276 | 36.15 | <0.0001 | <0.0001 | 1.5 |
| 30min | 20 | 0.725 | 0.393 | 28.04 | <0.0001 | <0.0001 | 2.4 |
| 30min | 30 | 0.769 | 0.482 | 23.96 | <0.0001 | <0.0001 | 3.7 |
| 1h | 10 | 0.694 | 0.296 | 24.36 | <0.0001 | <0.0001 | 1.5 |
| 1h | 20 | 0.750 | 0.414 | 19.49 | <0.0001 | <0.0001 | 2.6 |
| 1h | 30 | 0.784 | 0.494 | 16.71 | <0.0001 | <0.0001 | 3.5 |
| 2h | 10 | 0.667 | 0.281 | 17.26 | <0.0001 | <0.0001 | 1.6 |
| 2h | 20 | 0.739 | 0.385 | 14.90 | <0.0001 | <0.0001 | 2.9 |
| 2h | 30 | 0.792 | 0.475 | 13.16 | <0.0001 | <0.0001 | 4.5 |
| 4h | 10 | 0.650 | 0.280 | 11.80 | <0.0001 | <0.0001 | 1.8 |
| 4h | 20 | 0.692 | 0.380 | 9.37 | <0.0001 | <0.0001 | 2.5 |
| 4h | 30 | 0.738 | 0.470 | 7.90 | <0.0001 | <0.0001 | 3.8 |

Crossovers reach EMA in 1.5-4.5 bars on average. Both z-test (reach rate) and Mann-Whitney (time to reach) significant across all tests.

## BTC Futures(CME) - CONTROL B (vs Similar Moves)

In [ ]:
for tf_name, df in timeframes.items():
    print(f"\n=== {tf_name} ({df.shape[0]} bars) ===")
    for mb in [10, 20, 30]:
        cross_rate, control_rate, control_n = h2_control_b(df, max_bars=mb)
        print(f"{tf_name} mb={mb}: crossover={cross_rate:.3f}, similar_move={control_rate:.3f}, control_n={control_n}")

### BTC Control B Results

| Timeframe | Max Bars | Crossover Rate | Similar Move Rate | Control N |
|-----------|----------|---------------|-------------------|-----------|
| 15min | 10 | 0.663 | 0.255 | 8869 |
| 15min | 20 | 0.714 | 0.361 | 8868 |
| 15min | 30 | 0.758 | 0.443 | 8867 |
| 30min | 10 | 0.677 | 0.272 | 4225 |
| 30min | 20 | 0.725 | 0.373 | 4225 |
| 30min | 30 | 0.769 | 0.448 | 4223 |
| 1h | 10 | 0.695 | 0.275 | 2066 |
| 1h | 20 | 0.751 | 0.397 | 2065 |
| 1h | 30 | 0.785 | 0.473 | 2064 |
| 2h | 10 | 0.667 | 0.251 | 1173 |
| 2h | 20 | 0.738 | 0.352 | 1175 |
| 2h | 30 | 0.791 | 0.456 | 1174 |
| 4h | 10 | 0.649 | 0.259 | 669 |
| 4h | 20 | 0.693 | 0.383 | 669 |
| 4h | 30 | 0.739 | 0.470 | 668 |

Crossovers beat momentum-matched bars by 25-40 percentage points. The effect is not explained by move size alone.

## EUR/USD FUTURES(CME) - CONTROL A (vs Random Bars)

In [ ]:
for tf_name, df in eur_timeframes.items():
    for mb in [10, 20, 30]:
        real_rate, random_rate, z_score, z_p, mw_p, avg_bars = h2_test(df, max_bars=mb)
        print(f"{tf_name} mb={mb}: real={real_rate:.3f}, random={random_rate:.3f}, z={z_score:.2f}, z_p={z_p:.4f}, mw_p={mw_p:.4f}, avg_bars={avg_bars:.1f}")


### EUR/USD Control A Results

| Timeframe | Max Bars | Crossover Rate | Random Rate | Z-score | Z p-value | MW p-value | Avg Bars |
|-----------|----------|---------------|-------------|---------|-----------|------------|----------|
| 15min | 10 | 0.691 | 0.310 | 39.23 | <0.0001 | <0.0001 | 1.5 |
| 15min | 20 | 0.741 | 0.411 | 32.60 | <0.0001 | <0.0001 | 2.4 |
| 15min | 30 | 0.783 | 0.487 | 28.98 | <0.0001 | <0.0001 | 3.7 |
| 30min | 10 | 0.695 | 0.292 | 29.55 | <0.0001 | <0.0001 | 1.5 |
| 30min | 20 | 0.755 | 0.402 | 24.46 | <0.0001 | <0.0001 | 2.6 |
| 30min | 30 | 0.787 | 0.487 | 20.60 | <0.0001 | <0.0001 | 3.5 |
| 1h | 10 | 0.711 | 0.275 | 22.97 | <0.0001 | <0.0001 | 1.5 |
| 1h | 20 | 0.753 | 0.386 | 18.21 | <0.0001 | <0.0001 | 2.3 |
| 1h | 30 | 0.787 | 0.468 | 15.61 | <0.0001 | <0.0001 | 3.3 |
| 2h | 10 | 0.701 | 0.272 | 15.91 | <0.0001 | <0.0001 | 1.6 |
| 2h | 20 | 0.750 | 0.383 | 12.76 | <0.0001 | <0.0001 | 2.5 |
| 2h | 30 | 0.784 | 0.480 | 10.43 | <0.0001 | <0.0001 | 3.5 |
| 4h | 10 | 0.672 | 0.256 | 11.48 | <0.0001 | <0.0001 | 1.7 |
| 4h | 20 | 0.747 | 0.368 | 9.73 | <0.0001 | <0.0001 | 3.1 |
| 4h | 30 | 0.787 | 0.462 | 8.18 | <0.0001 | <0.0001 | 4.2 |

Nearly identical to BTC. Cross-asset confirmation with both statistical tests.

In [ ]:
for tf_name, df in eur_timeframes.items():
    print(f"\n=== {tf_name} ({df.shape[0]} bars) ===")
    for mb in [10, 20, 30]:
        cross_rate, control_rate, control_n = h2_control_b(df, max_bars=mb)
        print(f"{tf_name} mb={mb}: crossover={cross_rate:.3f}, similar_move={control_rate:.3f}, control_n={control_n}")

### EUR/USD Control B Results

| Timeframe | Max Bars | Crossover Rate | Similar Move Rate | Control N |
|-----------|----------|---------------|-------------------|-----------|
| 15min | 10 | 0.691 | 0.318 | 6185 |
| 15min | 20 | 0.741 | 0.421 | 6205 |
| 15min | 30 | 0.782 | 0.492 | 6209 |
| 30min | 10 | 0.695 | 0.298 | 2955 |
| 30min | 20 | 0.754 | 0.400 | 2952 |
| 30min | 30 | 0.786 | 0.473 | 2950 |
| 1h | 10 | 0.710 | 0.287 | 1476 |
| 1h | 20 | 0.753 | 0.393 | 1474 |
| 1h | 30 | 0.786 | 0.472 | 1471 |
| 2h | 10 | 0.700 | 0.279 | 843 |
| 2h | 20 | 0.748 | 0.371 | 844 |
| 2h | 30 | 0.782 | 0.464 | 843 |
| 4h | 10 | 0.672 | 0.223 | 462 |
| 4h | 20 | 0.747 | 0.364 | 462 |
| 4h | 30 | 0.785 | 0.445 | 463 |

Same 25-45pp gap. Effect holds across asset classes.

## H2 Conclusion

**30/30 tests significant (p < 0.0001) on both z-test and Mann-Whitney across both assets.** After dual SSL regime alignment, price reaches EMA 200 at 65-79% vs 25-49% from random bars, typically within 1.5-4.5 bars.The HMA regime crossover captures genuine directional structure that predicts both the likelihood and speed of movement toward the slow average. This is consistent across all timeframes and both crypto and FX markets.